# Autonomous Car Racing — SAC on CarRacing-v3 (Colab demo)

Runtime → *Change runtime type* → **GPU**. This notebook installs the project, trains the PPO profile,
evaluates it against baselines on held-out tracks, and shows videos inline (no GUI window needed).

> A few thousand steps will **not** produce a good driver. For real results raise `TIMESTEPS` (≥ 300k) – the run
> checkpoints regularly and can be resumed. Nothing in this notebook is pre-computed: every number comes from the cells you run.

## 1. Get the code and install
Upload / clone the repository so that `requirements.txt` is in the working directory.

In [ ]:
# Option A: clone your fork   ->   !git clone <your-repo-url> autonomous-car-racing
# Option B: upload autonomous-car-racing.zip via the Files sidebar and unzip:
# !unzip -q autonomous-car-racing.zip
%cd autonomous-car-racing

In [ ]:
!apt-get -qq install -y swig > /dev/null      # needed only if Box2D has to be compiled
!pip -q install -r requirements.txt
!python scripts/setup_environment.py

## 2. Hardware check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
# NOTE: the Box2D simulator runs on the CPU either way; the GPU accelerates the network updates.

## 3. Train (PPO profile that learned to follow the road)
Colab's free CPU count is small, so ~200k steps takes about 1.5-2 h. Use TIMESTEPS=40000 for a quick check (~15 min).

In [ ]:
TIMESTEPS = 200_000     # ~95 min on 2 CPU cores; Colab is similar (the simulator runs on the CPU)
RUN = "colab_ppo"
!python -m src.train --config configs/ppo_fast.yaml --run-name {RUN} --timesteps {TIMESTEPS} --device cpu


## 4. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 5. Evaluate vs. baselines on held-out tracks (same tracks for every policy)

In [ ]:
# the PPO policy is trained with action noise -> evaluate with sampled actions
# pick a checkpoint from checkpoints/{RUN}/ (e.g. ckpt_160000_steps.zip) or use best_model.zip
!python -m src.evaluate --model models/{RUN}/best_model.zip --episodes 10 --stochastic --compare-baselines


In [ ]:
from IPython.display import Image, display
display(Image(f"results/{RUN}/training_curves.png"))
display(Image(f"results/{RUN}/eval_best_model_comparison.png"))


## 6. Record and watch the driving (headless)

In [ ]:
!python -m src.inference --model models/{RUN}/best_model.zip --stochastic --episodes 1 --video videos/agent.mp4
!python -m src.inference --random --episodes 1 --video videos/random.mp4


In [ ]:
from IPython.display import Video, display
display(Video("videos/agent.mp4", embed=True, width=600))
display(Video("videos/random.mp4", embed=True, width=600))

## 7. Resume training later
```
!python -m src.train --resume checkpoints/colab_ppo --timesteps 100000
```
Checkpoints live in `checkpoints/<run>/`; copy them to Google Drive if the Colab session may be recycled.